In [ ]:
# Auto-injected: export figures as PNG + PDF + SVG into png/ pdf/ svg/
import os
import matplotlib.pyplot as plt
for _d in ('pdf', 'svg', 'png'):
    os.makedirs(_d, exist_ok=True)
def _save_fig(base, fig=None, dpi=300, **kw):
    f = fig if fig is not None else plt.gcf()
    f.savefig(f'pdf/{base}.pdf', bbox_inches='tight', **kw)
    f.savefig(f'svg/{base}.svg', bbox_inches='tight', **kw)
    f.savefig(f'png/{base}.png', dpi=dpi, bbox_inches='tight', **kw)


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import make_interp_spline
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

In [ ]:
df = pd.read_csv('../result/output_scenario.csv')
df['pop'] = np.where(df['pop'] > 500, 500, df['pop'])
df['total_cost_0_ave'] = df['total_cost_0'] / df['pop']
df['total_cost_2020_ave'] = df['total_cost_2020'] / df['pop']
df['total_cost_2050_ave'] = df['total_cost_2050'] / df['pop']
df['diff'] = (df['total_cost_2020_ave'] - df['total_cost_0_ave']) / df['total_cost_0_ave']
df

In [ ]:
print(df['total_cost_0_ave'].describe())

In [ ]:
print(df['total_cost_0_ave'].quantile([0.1, 0.25, 0.5, 0.75, 0.9,     
  0.95, 0.99]))

In [ ]:
print(df['diff'].describe())

In [ ]:
def visualize_cost(data, ipcc_regions, vmax, column):
    plt.rcParams['font.family'] = 'Arial'
    # 导入必要的模块
    from matplotlib.offsetbox import OffsetImage, AnnotationBbox
    import numpy as np
    from scipy.ndimage import gaussian_filter
    
    # 创建Point几何图形
    data['geometry'] = data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
    geo_df = gpd.GeoDataFrame(data, geometry='geometry')
    geo_df.set_crs(epsg=4326, inplace=True)
    
    # 创建带有cartopy投影的图形
    fig = plt.figure(figsize=(14, 6), dpi=500)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    ax.set_facecolor("#FFFFFF")  
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.2)
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, alpha=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
    
    if ipcc_regions is not None and not ipcc_regions.empty:
        # 遍历每个IPCC区域
        for _, region_row in ipcc_regions.iterrows():
            # (A) 绘制区域轮廓线
            ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor='none',      # 设置为'none'以只显示边框
                              edgecolor='#555555',     # 边框颜色
                              linewidth=0.8,         # 边框线宽
                              linestyle='--',         # 虚线样式
                              zorder=1)              # zorder确保它在密度图之上

            # (B) 添加区域标签
            # if region_row['geometry'] is not None and not region_row['geometry'].is_empty:
            #     # 为了防止标签位置在奇怪的地方（例如多边形集合的几何中心在海上）
            #     # 我们选择最大的那块多边形的中心点来放置标签
            #     geom = region_row['geometry']
            #     if geom.geom_type == 'MultiPolygon':
            #         geom = max(geom.geoms, key=lambda p: p.area)
                
            #     centroid = geom.centroid
                
            #     # 检查中心点是否确实在多边形内部
            #     if geom.contains(centroid):
            #         ax.text(centroid.x, centroid.y, region_row['Acronym'],
            #                 ha='center', va='center',
            #                 fontsize=12, color="#000000", weight='bold',
            #                 transform=ccrs.PlateCarree(),
            #                 zorder=11) # zorder设得很高，确保标签在所有元素的最上层
                    
    # 使用更简单的方法创建密度图
    resolution = 1.0  # 1度分辨率
    lons = np.arange(-180, 181, resolution)
    lats = np.arange(-90, 91, resolution)
    
    # 创建网格点
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    density_grid = np.zeros_like(lon_grid)
    
    # 提取所有点的经纬度和值
    all_lons = geo_df.geometry.x.values
    all_lats = geo_df.geometry.y.values
    all_values = geo_df[column].values
    
    # 使用更适合地理数据的影响半径
    bandwidth = 2.5
    
    # 计算每个网格点的密度值
    for i, row in enumerate(all_lons):
        lon, lat = all_lons[i], all_lats[i]
        value = all_values[i]
        
        # 计算此数据点对网格的影响
        for x in range(len(lats)):
            for y in range(len(lons)):
                grid_lon = lons[y]
                grid_lat = lats[x]
                
                # 计算距离
                dist = np.sqrt((grid_lon - lon)**2 + (grid_lat - lat)**2)
                
                # 只在一定半径内影响网格点
                if dist <= 3 * bandwidth:
                    # 使用高斯核计算权重
                    weight = np.exp(-0.5 * (dist / bandwidth)**2) * value
                    density_grid[x, y] += weight
    
    # 对密度图应用轻微的高斯平滑
    density_grid = gaussian_filter(density_grid, sigma=1.0)
    
    # 设置阈值，只在密度足够高的区域绘制斜线
    threshold = np.max(density_grid) * 0.2
    mask = density_grid < threshold
    
    # 设置上限值，避免极端值影响色彩分布
    # 使用百分位数而不是最大值来设置上限，避免离群值的影响
    upper_limit = np.percentile(density_grid[~mask], 99)  # 使用99%分位数作为上限
    
    # 裁剪过高的值
    density_grid_capped = np.clip(density_grid, 0, upper_limit)
    
    # 创建掩码数组
    density_grid_masked = np.ma.array(density_grid_capped, mask=mask)
    
    # 热力图颜色 - 使用更多颜色变化
    heatmap_colors = ['#012f4830', '#669aba40', '#fbf0d950', '#be142060', '#7a010170']
    heatmap_cmap = mpl.colors.LinearSegmentedColormap.from_list('heatmap_palette', heatmap_colors)
    
    # 创建自定义规范化对象，确保色彩分布均匀
    heatmap_norm = mpl.colors.Normalize(vmin=threshold, vmax=upper_limit)
    
    # 绘制密度图 - 只在有数据的区域绘制斜线
    contour = ax.contourf(
        lon_grid, lat_grid, density_grid_masked,
        transform=ccrs.PlateCarree(),
        cmap=heatmap_cmap,
        norm=heatmap_norm,  # 使用自定义规范化
        levels=8,
        alpha=0.7,
        zorder=2,
        # hatches=['/////', '/////', '/////', '/////', '/////', '/////', '/////', '/////'],
        extend='neither'
    )
    
    # 设置颜色映射
    sci_colors = ['#012f48', '#669aba', '#fbf0d9', '#be1420', '#7a0101']  # From blue to red
    custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('sci_palette', sci_colors)
    norm = mpl.colors.Normalize(vmin=300, vmax=vmax)
    
    # 计算点大小范围
    min_size = 20
    max_size = 200
    
    # 对数据按值排序，保证小值在下层，大值在上层
    geo_df = geo_df.sort_values(by=column)
    
    # 创建一个假散点用于颜色条
    fake_scatter = ax.scatter(
        [-1000], [-1000],  # 不可见位置
        c=[0],
        cmap=custom_cmap,
        vmin=300,
        vmax=vmax,
        s=1
    )
    
    # 为每个点创建并添加标记
    for idx, row in geo_df.iterrows():
        # 获取经纬度和值
        lon, lat = row.geometry.x, row.geometry.y
        value = row[column]
        
        # 计算点大小
        size = min_size + ((value / vmax) * (max_size - min_size))
        
        # 计算颜色
        color = custom_cmap(norm(value))
        
        # 创建临时图形来生成点图像
        temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi= 300)
        temp_fig.patch.set_alpha(0)  # 透明背景
        
        temp_ax = temp_fig.add_subplot(111)
        temp_ax.set_aspect('equal')
        temp_ax.patch.set_alpha(0)
        
        # 先绘制一个稍大的黑色圆作为边框
        outer_circle = plt.Circle(
            (0.5, 0.5),
            0.18,         # 稍大的半径
            color='black',  # 黑色
            alpha=1
        )
        temp_ax.add_patch(outer_circle)

        # 再绘制一个内圆作为主要颜色区域
        inner_circle = plt.Circle(
            (0.5, 0.5),
            0.15,         # 稍小的半径
            color=color,  # 数据颜色
            alpha=1
        )
        temp_ax.add_patch(inner_circle)
        
        # 设置坐标轴范围
        temp_ax.set_xlim(0, 1)
        temp_ax.set_ylim(0, 1)
        temp_ax.axis('off')  # 隐藏坐标轴
        
        # 渲染并获取图像
        temp_fig.tight_layout(pad=0)
        temp_fig.canvas.draw()
        point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
        plt.close(temp_fig)
        
        # 将地理坐标转换为投影坐标
        x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
        
        # 计算缩放因子 - 调整点大小
        zoom_factor = np.sqrt(size) / 250
        
        # 创建OffsetImage
        imagebox = OffsetImage(point_img, zoom=zoom_factor)
        imagebox.image.axes = ax
        
        # 创建并添加AnnotationBbox
        ab = AnnotationBbox(
            imagebox,
            (x, y),
            frameon=False,
            pad=0,
            zorder=10  # 确保点在最上层
        )
        ax.add_artist(ab)
    
    # 添加颜色条
    cbar = fig.colorbar(
        fake_scatter,
        ax=ax,
        orientation='horizontal',
        shrink=0.6,  # 控制颜色条长度
        pad=0.07,    # 调整颜色条和图形之间的间距
        aspect=50    # 控制颜色条的宽度（值越小越宽）
    )
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label(f'Per Capita Investment(USD)', fontsize=24)
    
    # 设置全球边界
    ax.set_global()
    # --- Longitude/latitude graticule (Nature Comms checklist) ---
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                      xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 16, 'color': '#404040'}
    gl.ylabel_style = {'size': 16, 'color': '#404040'}
     
    # 移除坐标轴刻度和边框
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.axis('off')  # 完全关闭坐标轴
    
    # 减少边距
    plt.tight_layout(pad=0)
    
    _save_fig('fig1_1a_per_capita_investment_map', fig)
    plt.show()

In [ ]:
vmax = 1e3
ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")
visualize_cost(df,ipcc_regions, vmax, 'total_cost_0_ave')
# visualize_cost(df, vmax, 'total_cost_2020_ave')
# visualize_cost(df, vmax, 'total_cost_2050_ave')

In [ ]:
def visualize_diff(data, vmax, column):
    plt.rcParams['font.family'] = 'Arial'
    # 导入必要的模块
    from matplotlib.offsetbox import OffsetImage, AnnotationBbox
    import numpy as np
    from scipy.ndimage import gaussian_filter
    
    # 创建Point几何图形
    data['geometry'] = data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
    geo_df = gpd.GeoDataFrame(data, geometry='geometry')
    geo_df.set_crs(epsg=4326, inplace=True)
    
    # 创建带有cartopy投影的图形
    fig = plt.figure(figsize=(14, 6), dpi=500)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    ax.set_facecolor('#CCCCCC')  
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
    
    # 使用更简单的方法创建密度图
    resolution = 1.0  # 1度分辨率
    lons = np.arange(-180, 181, resolution)
    lats = np.arange(-90, 91, resolution)
    
    # 创建网格点
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    density_grid = np.zeros_like(lon_grid)
    
    # 提取所有点的经纬度和值
    all_lons = geo_df.geometry.x.values
    all_lats = geo_df.geometry.y.values
    all_values = geo_df[column].values
    
    # 使用更适合地理数据的影响半径
    bandwidth = 3.0
    
    # 计算每个网格点的密度值
    for i, row in enumerate(all_lons):
        lon, lat = all_lons[i], all_lats[i]
        value = all_values[i]
        
        # 计算此数据点对网格的影响
        for x in range(len(lats)):
            for y in range(len(lons)):
                grid_lon = lons[y]
                grid_lat = lats[x]
                
                # 计算距离
                dist = np.sqrt((grid_lon - lon)**2 + (grid_lat - lat)**2)
                
                # 只在一定半径内影响网格点
                if dist <= 3 * bandwidth:
                    # 使用高斯核计算权重
                    weight = np.exp(-0.5 * (dist / bandwidth)**2) * value
                    density_grid[x, y] += weight
    
    # 对密度图应用轻微的高斯平滑
    density_grid = gaussian_filter(density_grid, sigma=1.0)
    
    # 设置阈值，只在密度足够高的区域绘制斜线
    threshold = np.max(density_grid) * 0.15
    mask = density_grid < threshold
    
    # 设置上限值，避免极端值影响色彩分布
    # 使用百分位数而不是最大值来设置上限，避免离群值的影响
    upper_limit = np.percentile(density_grid[~mask], 99)  # 使用99%分位数作为上限
    
    # 裁剪过高的值
    density_grid_capped = np.clip(density_grid, 0, upper_limit)
    
    # 创建掩码数组
    density_grid_masked = np.ma.array(density_grid_capped, mask=mask)
    
    # 热力图颜色 - 使用更多颜色变化
    heatmap_colors = ['#012f4830', '#669aba40', '#fbf0d950', '#be142060', '#7a010170']
    heatmap_cmap = mpl.colors.LinearSegmentedColormap.from_list('heatmap_palette', heatmap_colors)
    
    # 创建自定义规范化对象，确保色彩分布均匀
    heatmap_norm = mpl.colors.Normalize(vmin=threshold, vmax=upper_limit)
    
    # 绘制密度图 - 只在有数据的区域绘制斜线
    contour = ax.contourf(
        lon_grid, lat_grid, density_grid_masked,
        transform=ccrs.PlateCarree(),
        cmap=heatmap_cmap,
        norm=heatmap_norm,  # 使用自定义规范化
        levels=8,
        alpha=0.7,
        zorder=2,
        # hatches=['/////', '/////', '/////', '/////', '/////', '/////', '/////', '/////'],
        extend='neither'
    )
    
    # 设置颜色映射
    sci_colors = ['#012f48', '#669aba', '#fbf0d9', '#be1420', '#7a0101']  # From blue to red
    custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('sci_palette', sci_colors)
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)
    
    # 计算点大小范围
    min_size = 40
    max_size = 200
    
    # 对数据按值排序，保证小值在下层，大值在上层
    geo_df = geo_df.sort_values(by=column)
    
    # 创建一个假散点用于颜色条
    fake_scatter = ax.scatter(
        [-1000], [-1000],  # 不可见位置
        c=[0],
        cmap=custom_cmap,
        vmin=-vmax,
        vmax=vmax,
        s=1
    )
    
    # 为每个点创建并添加标记
    for idx, row in geo_df.iterrows():
        # 获取经纬度和值
        lon, lat = row.geometry.x, row.geometry.y
        value = row[column]
        
        # 计算点大小
        size = min_size + ((value / vmax) * (max_size - min_size))
        
        # 计算颜色
        color = custom_cmap(norm(value))
        
        # 创建临时图形来生成点图像
        temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi= 300)
        temp_fig.patch.set_alpha(0)  # 透明背景
        
        temp_ax = temp_fig.add_subplot(111)
        temp_ax.set_aspect('equal')
        temp_ax.patch.set_alpha(0)
        
        # 先绘制一个稍大的黑色圆作为边框
        outer_circle = plt.Circle(
            (0.5, 0.5),
            0.18,         # 稍大的半径
            color='black',  # 黑色
            alpha=1
        )
        temp_ax.add_patch(outer_circle)

        # 再绘制一个内圆作为主要颜色区域
        inner_circle = plt.Circle(
            (0.5, 0.5),
            0.15,         # 稍小的半径
            color=color,  # 数据颜色
            alpha=1
        )
        temp_ax.add_patch(inner_circle)
        
        # 设置坐标轴范围
        temp_ax.set_xlim(0, 1)
        temp_ax.set_ylim(0, 1)
        temp_ax.axis('off')  # 隐藏坐标轴
        
        # 渲染并获取图像
        temp_fig.tight_layout(pad=0)
        temp_fig.canvas.draw()
        point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
        plt.close(temp_fig)
        
        # 将地理坐标转换为投影坐标
        x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
        
        # 计算缩放因子 - 调整点大小
        zoom_factor = np.sqrt(size) / 250
        
        # 创建OffsetImage
        imagebox = OffsetImage(point_img, zoom=zoom_factor)
        imagebox.image.axes = ax
        
        # 创建并添加AnnotationBbox
        ab = AnnotationBbox(
            imagebox,
            (x, y),
            frameon=False,
            pad=0,
            zorder=10  # 确保点在最上层
        )
        ax.add_artist(ab)
    
    # 添加颜色条
    cbar = fig.colorbar(
        fake_scatter,
        ax=ax,
        orientation='horizontal',
        shrink=0.6,  # 控制颜色条长度
        pad=0.03,    # 调整颜色条和图形之间的间距
        aspect=50    # 控制颜色条的宽度（值越小越宽）
    )
    cbar.set_label(f'Diff', fontsize=12)
    
    # 设置全球边界
    ax.set_global()
    # --- Longitude/latitude graticule (Nature Comms checklist) ---
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                      xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 16, 'color': '#404040'}
    gl.ylabel_style = {'size': 16, 'color': '#404040'}
     
    # 移除坐标轴刻度和边框
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.axis('off')  # 完全关闭坐标轴
    
    # 减少边距
    plt.tight_layout(pad=0)
    
    _save_fig('fig1_1b_cost_diff_map', fig)
    plt.show()

In [ ]:
vmax = 0.5
# visualize_diff(df, vmax, 'diff')

In [ ]:
from matplotlib.ticker import ScalarFormatter

def create_longitude_profile(data, column, ax=None):
    plt.rcParams['font.family'] = 'Arial'
    """创建经度方向的曲线图"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 1.5),dpi=500)
    
    # 按经度分组计算sum值
    lon_bins = np.arange(-180, 181, 5)  # 5度一组
    lon_centers = (lon_bins[:-1] + lon_bins[1:]) / 2
    lon_values = []
    
    for i in range(len(lon_bins)-1):
        mask = (data['lon'] >= lon_bins[i]) & (data['lon'] < lon_bins[i+1])
        if mask.any():
            lon_values.append(data.loc[mask, column].mean())
        else:
            lon_values.append(0)  # 用0代替NaN
    
    # 平滑曲线
    if len(lon_centers) > 3:
        lon_smooth = np.linspace(min(lon_centers), max(lon_centers), 300)
        spline = make_interp_spline(lon_centers, lon_values, k=3)
        lon_values_smooth = spline(lon_smooth)
        # 确保没有负值
        lon_values_smooth = np.maximum(lon_values_smooth, 0)
    else:
        lon_smooth = lon_centers
        lon_values_smooth = lon_values
    
    # 绘制曲线
    ax.plot(lon_smooth, lon_values_smooth, color='#7a0101', linewidth=1.5)
    ax.fill_between(lon_smooth, 0, lon_values_smooth, alpha=0.3, color="#a37070")
    
    # 设置范围和网格线
    ax.set_xlim(-180, 180)
    ax.set_ylim(0, None)
    ax.tick_params(axis='y', labelsize=24) 
    
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    ax.yaxis.offsetText.set_fontsize(18) 
    # 美化图表
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    # ax.set_ylabel('Avg. Investment', fontsize=9)
    
    return ax

def create_latitude_profile(data, column, ax=None):
    """创建纬度方向的曲线图"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(2, 10),dpi=500)
    
    # 按纬度分组计算sum值
    lat_bins = np.arange(-90, 91, 2)  # 2度一组
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    lat_values = []
    
    for i in range(len(lat_bins)-1):
        mask = (data['lat'] >= lat_bins[i]) & (data['lat'] < lat_bins[i+1])
        if mask.any():
            lat_values.append(data.loc[mask, column].mean())
        else:
            lat_values.append(0)  # 用0代替NaN
    
    # 平滑曲线
    if len(lat_centers) > 3:
        lat_smooth = np.linspace(min(lat_centers), max(lat_centers), 300)
        spline = make_interp_spline(lat_centers, lat_values, k=3)
        lat_values_smooth = spline(lat_smooth)
        # 确保没有负值
        lat_values_smooth = np.maximum(lat_values_smooth, 0)
    else:
        lat_smooth = lat_centers
        lat_values_smooth = lat_values
    
    # 绘制曲线
    ax.plot(lat_values_smooth, lat_smooth, color='#7a0101', linewidth=1.5)
    ax.fill_betweenx(lat_smooth, 0, lat_values_smooth, alpha=0.3, color='#a37070')
    
    # 设置范围和网格线
    ax.set_ylim(-90, 90)
    ax.set_xlim(0, None)
    ax.tick_params(axis='x', labelsize=24) 
    
    ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))
    ax.xaxis.offsetText.set_fontsize(18)
    # 美化图表
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # ax.set_xlabel('Avg. Investment', fontsize=9)
    
    return ax

In [ ]:
ax_lon = create_longitude_profile(df, 'total_cost_0_ave', ax=None)
_save_fig('fig1_1c_longitude_profile', ax_lon.figure)
ax_lat = create_latitude_profile(df, 'total_cost_0_ave', ax=None)
_save_fig('fig1_1d_latitude_profile', ax_lat.figure)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. --- Global Settings ---

# Set global font and style for a professional look
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 24,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24, # Slightly larger for legend clarity
    'figure.titlesize': 24
})

# Define the mapping from CSV columns directly to publication-quality names
column_mapping = {
    'renewable_cost_per_capita': "Renewable Generation Cost",
    'storage_cost_per_capita': "Energy Storage System Cost",
    'lng_cost_per_capita': "LNG Cost",
    'other_equipment_cost_per_capita': "Sector-Coupling System Cost",
    'discard_cost_per_capita': "Cost of Energy Curtailment",
    'load_shedding_cost_per_capita': "Cost of Unserved Energy"
}

# Define the color for each cost type, using the publication-quality names as keys
cost_color_map = {
    "Renewable Generation Cost": "#4C72B0",
    "Energy Storage System Cost": "#009988",
    "LNG Cost": "#808080",
    "Sector-Coupling System Cost": "#8172B3",
    "Cost of Energy Curtailment": "#d47d49",
    "Cost of Unserved Energy": "#C44E52"
}

# The new_name_map dictionary is no longer needed.


# 2. --- Data Loading and Processing ---

# Load the island cost data
real_cost_df = pd.read_csv('../result/island_cost_summary_0.csv')

# Filter for regions with sufficient data
region_counts = real_cost_df['IPCC_Region_Code'].value_counts()
valid_regions = region_counts[region_counts >= 10].index.tolist()
filtered_df = real_cost_df[real_cost_df['IPCC_Region_Code'].isin(valid_regions)].copy()

# Rename columns for internal use using the direct mapping
for old_col, new_col in column_mapping.items():
    if old_col in filtered_df.columns:
        filtered_df[new_col] = filtered_df[old_col]

# Sort regions by their mean total per-capita cost
cost_names = list(column_mapping.values())
filtered_df['total_cost_per_capita'] = filtered_df[cost_names].sum(axis=1)
total_cost_per_region = filtered_df.groupby('IPCC_Region_Code')['total_cost_per_capita'].mean()
areas = total_cost_per_region.sort_values(ascending=False).index.tolist()

# Rename region column for plotting
filtered_df['Area'] = filtered_df['IPCC_Region_Code']

print(f"Using data from {len(filtered_df)} islands.")
print(f"Regions included: {areas}")
print(f"Cost types (formal names): {cost_names}")


# 3. --- Generate Individual Raincloud Plots ---

# Loop through each cost type to generate a separate plot
for cost_name in cost_names:
    # cost_name is now the formal name, e.g., "Renewable Generation CAPEX"
    current_color = cost_color_map[cost_name]
    
    # Prepare data for the current cost type, ensuring correct region order
    area_groups = filtered_df.groupby('Area')[cost_name].apply(list)
    area_groups = area_groups.reindex(areas)

    # Create figure and axes
    fig, ax = plt.subplots(figsize=(16, 5.8), dpi=300)

    # Define positions on the x-axis
    positions = np.arange(len(areas)) + 1

    # Create a combined raincloud plot for each region
    for i, area in enumerate(areas):
        if area not in area_groups.index or len(area_groups[area]) == 0:
            continue
            
        # A. Half-violin plot (the "cloud")
        violin_parts = ax.violinplot(
            area_groups[area], positions=[positions[i]],
            showmeans=False, showmedians=False, showextrema=False
        )
        
        # Modify the violin to show only the right half
        for pc in violin_parts['bodies']:
            vertices = pc.get_paths()[0].vertices
            center_x = positions[i]
            vertices[vertices[:, 0] < center_x, 0] = center_x
            pc.set_facecolor(current_color)
            pc.set_edgecolor('black')
            pc.set_alpha(0.8)

        # B. Boxplot (the "rain")
        ax.boxplot(
            area_groups[area], positions=[positions[i]], widths=0.18,
            patch_artist=True,
            boxprops=dict(facecolor='white', color='black'),
            medianprops=dict(color='#be1420', linewidth=3),
            whiskerprops=dict(color='black', linewidth=1),
            capprops=dict(color='black', linewidth=1),
            flierprops=dict(marker='o', color='black', markersize=4, markerfacecolor='gray'),
            showfliers=True
        )

        # C. Scatter plot (the "raindrops")
        jittered_x = np.random.normal(positions[i] - 0.15, 0.03, size=len(area_groups[area]))
        ax.scatter(
            jittered_x, area_groups[area], color=current_color,
            alpha=0.7, s=20, edgecolors='black', linewidths=0.5
        )

    # --- Formatting and Styling ---
    
    # The cost_name is now the formal name, so we use it directly for the y-axis label.
    ax.set_ylabel(f'{cost_name}')

    # Set x-axis ticks and labels
    ax.set_xticks(positions[:len(areas)])
    ax.set_xticklabels(areas, rotation=40, ha='right')

    # Clean up the plot appearance
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    _save_fig('fig1_1e_raincloud_' + cost_name.replace(' ', '_').replace('/', '_'), fig)
    plt.show()


# 4. --- Generate a Separate Legend ---

# Create a new figure specifically for the legend
fig_legend, ax_legend = plt.subplots(figsize=(14, 1.5), dpi=300)

# Create legend handles directly from the updated cost_color_map
legend_handles = [mpatches.Patch(color=color, label=label) 
                  for label, color in cost_color_map.items()]

# Add the legend to the figure
fig_legend.legend(
    handles=legend_handles, 
    loc='center', 
    frameon=False, 
    ncol=3 # Arrange in 3 columns for better fit
)

# Hide the axes of the legend figure
ax_legend.axis('off')
plt.tight_layout()
_save_fig('fig1_1e_raincloud_legend', fig_legend)
plt.show()

